In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB ---- collection 단위로 저장
- Pinecone : 클라우드 vector DB ---- index 단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env에 추가(PINECONE_API_KEY등록)


# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


# 1. knowlege base 구성을 위한 데이터 생성

In [7]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # character 단위로 짜름
loader = Docx2txtLoader('tax_docx/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, #문서를 쪼갤 떄 1500글자씩 chunking
        chunk_overlap=20, #200문자씩 overlap 하여 chunking 작업
        #separators=["\n\n","\n"," "] -> 기본값   : 다음의 순서대로 chunking 시도
)
documents = loader.load_and_split(text_splitter=text_splitter)
print('chunk 갯수: ', len(documents))

chunk 갯수:  180


In [9]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding= UpstageEmbeddings(model='solar-embedding-1-large-passage')

In [14]:
%%time
#pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
pc=Pinecone(#api_key=os.getenv('PINECONE_API_KEY')
            )
# 데이터를 처음 저장할 때 
index_name ='tax-index'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name,
)


CPU times: total: 15.6 ms
Wall time: 0 ns


# 2. 답변 생성을 위한 Retrieval

In [18]:
query="연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3)

In [30]:
retrieved_doc="\n--\n".join([doc.page_content.replace("\n\n","") for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [15]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model='gpt-4.1-nano')

In [32]:
from langchain_upstage import ChatUpstage
llm=ChatUpstage(model='solar-pro2')

In [34]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [35]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 **근로소득공제, 종합소득산출세액, 세액감면/공제** 등을 단계적으로 적용해야 합니다. 다음은 [context]에 제공된 소득세법 규정을 기반으로 한 계산 절차입니다.

---

### 1. **근로소득 공제** (소득세법 제59조 외 관련 규정)
- **총급여액**: 5,000만원  
- **근로소득 공제액**:  
  - 5,000만원 × 15% + 180만원 = **930만원**  
  - *계산식: 5,000만원 × 0.15 + 180만원 (5천만원 이하 구간 적용)*  
- **근로소득 금액**: 5,000만원 - 930만원 = **4,070만원**

---

### 2. **종합소득 과세표준 계산**
- **기본공제**: 본인 1명 × 150만원 = 150만원  
- **추가 공제** (예: 연금보험료, 의료비 등)는 [context]에 언급되지 않았으므로 제외.  
- **과세표준**: 4,070만원 - 150만원 = **3,920만원**

---

### 3. **종합소득 산출세액 계산** (2024년 세율 적용)
- **누진세율** 적용:  
  - 1,200만원 × 6% = 72만원  
  - (4,600만원 - 1,200만원) × 15% = 510만원  
  - *3,920만원은 1,200만원 초과 ~ 4,600만원 미만 구간에 해당하므로*  
  - **산출세액**: 72만원 + (3,920만원 - 1,200만원) × 15% = **72만원 + 408만원 = 480만원**

---

### 4. **세액공제 적용** ([context] 제59조, 제59조의4 등)
#### (1) **근로소득세액공제** (제59조 ②항)
- **총급여액 5,000만원**은 "3,300만원 초과 ~ 7,000만원 이하" 구간에 해당:  
  - 기본공제액: 74만원 - (5,000만원 - 3,300만원) × 0.8% = **74만원 - 14.4만원 = 59.6만원**  
  - *단, 최소 66만원 제한 적